## Preprocessing OpenNQ dataset into an evaluation split

In [1]:
import json, os, random

In [2]:
data =[]
with open('../data/raw/OpenNQ/nq-dev-all.jsonl', 'r') as file:
    for line in file:
        data.append(json.loads(line))
    

In [3]:
random.seed(42)

In [4]:
len(data)

7830

In [5]:
for i, datapoint in enumerate(data):
    datapoint['datapoint_id'] = i

In [6]:
data_with_short_answers = []
for datapoint in data:
    has_short_answer = True
    for annotation in datapoint['annotations']:
        if len(annotation['short_answers']) == 0 and annotation['yes_no_answer'] == 'NONE':
            has_short_answer = False
            break
    if has_short_answer:
        data_with_short_answers.append(datapoint)


In [7]:
len(data_with_short_answers)

1166

In [8]:
data_sample = random.sample(data_with_short_answers, 300)

In [9]:
def extract_short_answer(start_token, end_token, document_tokens):
    tokens =[]
    for i in range(start_token, end_token):
        tokens.append(document_tokens[i]['token'])
    return ' '.join(tokens)

In [10]:
data_sample[0]

{'annotations': [{'annotation_id': 1921617220728590481,
   'long_answer': {'candidate_index': 15,
    'end_byte': 49641,
    'end_token': 715,
    'start_byte': 48364,
    'start_token': 491},
   'short_answers': [{'end_byte': 48758,
     'end_token': 559,
     'start_byte': 48755,
     'start_token': 558}],
   'yes_no_answer': 'NONE'},
  {'annotation_id': 9207384796798450927,
   'long_answer': {'candidate_index': 15,
    'end_byte': 49641,
    'end_token': 715,
    'start_byte': 48364,
    'start_token': 491},
   'short_answers': [{'end_byte': 48758,
     'end_token': 559,
     'start_byte': 48755,
     'start_token': 558}],
   'yes_no_answer': 'NONE'},
  {'annotation_id': 16938966442939007844,
   'long_answer': {'candidate_index': 15,
    'end_byte': 49641,
    'end_token': 715,
    'start_byte': 48364,
    'start_token': 491},
   'short_answers': [{'end_byte': 48758,
     'end_token': 559,
     'start_byte': 48755,
     'start_token': 558}],
   'yes_no_answer': 'NONE'},
  {'annotati

In [11]:
sample_points = []
for i, datapoint in enumerate(data_sample):
    sample_point ={"question_id": i, "datapoint_id": datapoint['datapoint_id'], "question": datapoint['question_text'], "annotations": []}
    for j, annotation in enumerate(datapoint['annotations']):
        annotation_dict = {'short_answers': [], 'yes_no_answer': annotation['yes_no_answer']}
        for k, short_answer in enumerate(annotation['short_answers']):
            annotation_dict['short_answers'].append(extract_short_answer(short_answer['start_token'], short_answer['end_token'], datapoint['document_tokens']))
        sample_point['annotations'].append(annotation_dict)
    sample_points.append(sample_point)

In [13]:
with open('OpenNQ_staged.json', 'w') as file:
    json.dump(sample_points, file, indent=4)

After using the web utility, annotation have been picked. Now we handle problematic annotations and generate the final evaluation json

In [12]:
with open('OpenNQ_picked.json', 'r') as file:
    picked_data = json.load(file)

In [13]:
sample_points

[{'question_id': 0,
  'datapoint_id': 1452,
  'question': 'dogs name in the grinch who stole christmas',
  'annotations': [{'short_answers': ['Max'], 'yes_no_answer': 'NONE'},
   {'short_answers': ['Max'], 'yes_no_answer': 'NONE'},
   {'short_answers': ['Max'], 'yes_no_answer': 'NONE'},
   {'short_answers': ['Max'], 'yes_no_answer': 'NONE'},
   {'short_answers': ['Max'], 'yes_no_answer': 'NONE'}]},
 {'question_id': 1,
  'datapoint_id': 304,
  'question': 'where is the 7th game of the world series played',
  'annotations': [{'short_answers': ['at the site of the team holding the home advantage across the series'],
    'yes_no_answer': 'NONE'},
   {'short_answers': ['generally played at the site of the team holding the home advantage across the series'],
    'yes_no_answer': 'NONE'},
   {'short_answers': ['the site of the team holding the home advantage across the series'],
    'yes_no_answer': 'NONE'},
   {'short_answers': ['at the site of the team holding the home advantage across the 

In [14]:
picked_data

[{'question': 'dogs name in the grinch who stole christmas',
  'gt_answer': 'Max',
  'picked_annotation': [0, 'short_answer', 0]},
 {'question': 'where is the 7th game of the world series played',
  'gt_answer': 'at the site of the team holding the home advantage across the series',
  'picked_annotation': [0, 'short_answer', 0]},
 {'question': 'who designed the earth day flag in 1969',
  'gt_answer': 'John McConnell',
  'picked_annotation': [0, 'short_answer', 0]},
 {'question': 'when did sierra nevada brewery open in asheville',
  'gt_answer': 'early 2014',
  'picked_annotation': [2, 'short_answer', 0]},
 {'question': 'who is the actor that plays jt on the young and the restless',
  'gt_answer': 'Thaddeus Rowe Luckinbill',
  'picked_annotation': [0, 'short_answer', 0]},
 {'question': 'who plays sheila carter on the bold and the beautiful',
  'gt_answer': 'Kimberlin Brown',
  'picked_annotation': [0, 'short_answer', 0]},
 {'question': 'who sang what are we doing in love',
  'gt_answer'

In [15]:
final_dataset = []
for picked_datapoint, sample_point in zip(picked_data, sample_points):
    final_datapoint = {
        "question_id": sample_point['question_id'],
        "datapoint_id": sample_point['datapoint_id'],
        "question": sample_point['question'],
        "gt_answer": picked_datapoint['gt_answer'],
        "annotation_idx": picked_datapoint['picked_annotation'][0],
        "annotation_type": picked_datapoint['picked_annotation'][1],
        "short_answer_idx": picked_datapoint['picked_annotation'][2]
    }
    
    final_dataset.append(final_datapoint)

In [16]:
final_dataset

[{'question_id': 0,
  'datapoint_id': 1452,
  'question': 'dogs name in the grinch who stole christmas',
  'gt_answer': 'Max',
  'annotation_idx': 0,
  'annotation_type': 'short_answer',
  'short_answer_idx': 0},
 {'question_id': 1,
  'datapoint_id': 304,
  'question': 'where is the 7th game of the world series played',
  'gt_answer': 'at the site of the team holding the home advantage across the series',
  'annotation_idx': 0,
  'annotation_type': 'short_answer',
  'short_answer_idx': 0},
 {'question_id': 2,
  'datapoint_id': 3838,
  'question': 'who designed the earth day flag in 1969',
  'gt_answer': 'John McConnell',
  'annotation_idx': 0,
  'annotation_type': 'short_answer',
  'short_answer_idx': 0},
 {'question_id': 3,
  'datapoint_id': 3443,
  'question': 'when did sierra nevada brewery open in asheville',
  'gt_answer': 'early 2014',
  'annotation_idx': 2,
  'annotation_type': 'short_answer',
  'short_answer_idx': 0},
 {'question_id': 4,
  'datapoint_id': 3198,
  'question': 'w

In [20]:
None

In [17]:
trouble_indices = [45, 54, 76, 77, 122, 132, 134, 136, 177, 200, 219, 250, 266, 267]

In [18]:
for i in trouble_indices:
    print(final_dataset[i]['question'])

who does the voice of mickey mouse on mickey mouse clubhouse
make it or break it who goes to the olympics
who scored fastest 10000 runs in test cricket
who developed a set of postulates to prove that specific microorganisms cause disease
who stars in kevin probably save the world
who started the guinness book of world records
what movie is count on me by bruno mars in
who won every men's biathlon event in the 2002 winter olympics
when was the last time astros was in the world series
where are alkali metals located on the periodic table
how many episodes are there in modern family
when did frank sinatra first sing new york new york
when did the newest macbook pro come out
when did the golden state warriors win the finals


In [ ]:
answers = ["Wayne Allwine in Seasons 1–3 and Bret Iwan in Season 4",
"Payson, Lauren, Kaylie, Jordan, and Colleen",
"Brian Lara, Sachin Tendulkar and Kumar Sangakkara",
"Robert Koch and Friedrich Loeffler",
"Jason Ritter, JoAnna Garcia Swisher, Kimberly Hébert Gregory, India de Beaufort, J. August Richards, Chloe East, and Dustin Ybarra",
"Sir Hugh Beaver created the concept, and twin brothers Norris and Ross McWhirter co-founded the book",
"'Diary of a Wimpy Kid: The Long Haul' and 'A Turtle's Tale: Sammy's Adventures'",
"The Norwegian biathlete Ole Einar Bjørndalen",
"2022",
"Group 1 and the s-block",
"250",
"1978",
"October 30, 2024",
"1947, 1956, 1975, 2015, 2017, 2018, and 2022"]

In [24]:
len(answers)

14

In [26]:
for idx, answer in zip(trouble_indices, answers):
    final_dataset[idx]['gt_answer'] = answer
    final_dataset[idx]['annotation_idx'] = None
    final_dataset[idx]['annotation_type'] = 'manual'
    final_dataset[idx]['short_answer_idx'] = None


In [28]:
final_dataset[250]

{'question_id': 250,
 'datapoint_id': 4159,
 'question': 'when did frank sinatra first sing new york new york',
 'gt_answer': '1978',
 'annotation_idx': None,
 'annotation_type': 'manual',
 'short_answer_idx': None}

In [29]:
with open('../data/eval/OpenNQ.json', 'w') as file:
    json.dump(final_dataset, file, indent=4) 